# 14j — publication figures: contact degree distributions, and the fitted contact mean over time

This is the **publication** notebook. 8j–13j all answer *"is the fit healthy?"*; this one answers
*"what do the contact data look like, and what does the fitted model say the mean contact rate did?"*

| § | Question |
|---|---|
| §0 | Are the Stage-1 chains in `dt_intermediate/` compatible with the **current** model structure? |
| §1–§3 | The observed degree distribution (pdf + ccdf) over **two windows** — 2021-07-01 – 12-31 and 2021-10-01 – 12-31 — alongside the duration-weighted degree computed **with** and **without** group contacts, with a NegBin fitted to the counts and a hurdle-Weibull fitted to the weighted-with-group series. Six groups: the adult and child **marginals**, and the four **contactor → contactee** blocks (child→child … adult→adult) |
| §4 | Total fitted contacts **per age block** over time, from the NegBin and hurdle-Weibull Stage-1 chains, plus the hurdle-Weibull **excess (neighbourhood) degree** — the mean-NGM and neighbourhood-NGM inputs side by side |

**Definitions used throughout**, so all sections are mutually comparable:

- **child / adult** is the model's own block rule — `block_of(bin, cfg)` with `cfg.child_bins = 2`,
  i.e. child = CIS bins `2-10` + `11-15` (ages 2–15), adult = bins `16-24` … `70+` (16+). Ambiguous
  participant ages (CoMix's `12-17` group is the only one that straddles the cut) are resolved by
  `assign_age_bin`'s population-weighted draw — the same rule `prepare_degree_data` uses. The same
  rule bins **contactees**, so §3's four block pairs are exactly `model_degree`'s four dispersion
  blocks `bl = 2·(block_of(i)−1) + block_of(j)`.
- **duration weights** are `prepare_degree_data`'s: `duration_weight(duration_multi, cfg.d_max)` for
  individually-reported contacts, `cfg.w_dur_group` for group (`"mass"`) contacts, and **0** for group
  contacts in the "without" variant.

⚠ **This notebook never fits or writes a grid artefact.** Every Stage-1 reader is
`isfile(path) || return nothing` and nothing here calls `two_stage_forecast` / `fit_or_load_*`, which
fit on miss and write into `../dt_intermediate` under the live token. The only sampling done here is
§2's 2–3 parameter marginal fits, which go to `../res`.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # base + CoMix pipeline + forecasting framework
include("9j_viz_utils.jl")     # PERIODS + shade_periods! (the phase bands plot_wis_diff_over_time
                               #   uses); it includes 8j_viz_utils.jl, so 8j is loaded exactly once
include("10j_viz_utils.jl")    # reconstruct_mu_draws / reconstruct_p0_draws / _stage1_gp_generation
include("14j_viz_utils.jl")    # this notebook's helpers
using Random, Statistics, DataFrames, Printf, CSV
mkpath("../res")

In [ ]:
cfg  = FrameworkConfig(constant_contacts = false)   # production defaults: per-week contacts, NUTS
grid = cis_age_grid()

# 14j reads Stage-1 chains ONLY, and takes the framework default `stage1_use_nuts = true`, so —
# like 12j — it needs no `ENV["STAGE1_USE_NUTS"]`. Reading that variable would only create a way to
# point the notebook at a generation that was never fitted, which is how 9j/10j grind (they refit on
# miss). Assert instead.
@assert cfg.stage1_use_nuts
@assert contacts_label(cfg) == "temporal-w8h-lc0-nuts"

# §1–§3 are produced for EACH of these. The second is a sub-window of the first, so the pair shows
# how much of the July–December picture is carried by the last quarter alone.
WINDOWS = [(Date(2021, 7, 1), Date(2021, 12, 31)),
           (Date(2021, 10, 1), Date(2021, 12, 31))]
FIT_END = Date(2021, 12, 31)                      # matches 8j/9j, ⇒ last origin 2021-12-26

raw     = load_raw_contact_inputs()               # ONE read of both arrow files, reused below
ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw, origin_max = FIT_END)

@printf("token   %s\ngrid    %s\nwindows %s\norigins %d  (%s … %s)\n",
        contacts_label(cfg), join(grid.LAB, ", "),
        join(["$(a) … $(b)" for (a, b) in WINDOWS], " | "),
        length(ORIGINS), first(ORIGINS), last(ORIGINS))

## §0 — Are the chains on disk compatible with the current model?

`tmp/check_grid.jl`, which CLAUDE.md points at for this, left the tree along with the rest of `tmp/`.
`audit_stage1_grid` is its replacement and asks three separate questions, because they fail differently:

1. **Coverage** — which `(degree × origin × horizon)` cells exist. Missing cells are reported, not failed.
2. **Provenance** — every setting that changes the draws but is deliberately kept out of the cache
   token, so a partially-refitted grid is detectable at all: `sampler`, `ad_backend`, `target_accept`,
   `nuts_adapts`, `nuts_draws`, `phi_init_scale`, `phi_pf_max`. These must be **uniform**.
   `phi_pf_override` is tallied only — it is a per-cell *outcome* and varies by design.
3. **Structure** — the parameter block against the current `model_degree`: `z` rows = 27 (`-s0`),
   `z` cols = `Tn = n_fit + h` (`-w8h`), `z_c` = `Tn − 1` (`-t0`), dispersion `4·Tn`, `p0f` = `A²·Tn`
   on the weighted path, total `5 + 32·Tn` / `5 + 81·Tn`; plus the three generation gates
   (`-diag` refusal, `phi_time`-vs-`log_rho_time` fork, pre-`-m32` token refusal) reused from
   `_stage1_gp_generation` rather than restated.

`structural = :sample` checks one chain per degree × horizon (a full deserialise is ~3 s each);
pass `:all` for every file. The report says to escalate if provenance turns out mixed.

⚠ `frac_at_max_depth` is reported but is **not** a failure condition, and is easy to misread:
`_nuts_diagnostics` computes it at the chain's *observed* maximum depth, so `1.0` at depth 7 under a
cap of 10 means the sampler settled at a constant tree depth — healthy — not that it saturated.

⚠ **A shape audit cannot see a content defect.** See the note under §4: one chain once arrived with
part of its `p0f` array unwritten, passing `isfile`, the structural check and the generation gates
alike. That class of failure is caught in §4 by `_p0_draws_valid`, not here.

In [ ]:
AUDIT = audit_stage1_grid(cfg; grid = grid, craw = raw.craw, origins = ORIGINS,
                          structural = :sample)
AUDIT.verdict || @warn "Stage-1 grid did not pass cleanly — read the report above before trusting §4"
AUDIT.coverage

### Reading the §0 report

At the time of writing, the grid is **complete** and the audit reports:

- **504 of 504 cells present** — both degree models, all 63 origins × h1–4, 2020-10-18 … 2021-12-26.
  (An earlier partial state had 321 cells with the hurdle-Weibull path reaching only 2021-02-14; that
  is why §4's helpers tolerate gaps.) There are also 1512 Stage-2 artefacts now, which 14j does not
  use — it is a Stage-1 notebook by design, and 9j is where the transmission diagnostics live.
- **Provenance uniform** across all 504: `nuts / mooncake / 0.95 / 1000 adapts / 2000 draws /
  phi_init_scale 0.1`.
- **`phi_pf_max` and `phi_pf_override` absent on every file** (`:ABSENT`). These keys arrived with the
  2026-08-11 φ-boundary guard, so this grid predates it. It is a NUTS *starting value* setting, not a
  model property — the same class as `phi_init_scale` — and it is absent *uniformly*, so the grid is
  one generation and the chains are interchangeable. The mixed-provenance risk of a part-refitted grid
  did not materialise.
- **Zero divergences**, min ESS 142, maximum tree depth 9 against a cap of 10.

## §1 — Observed contact degree

`marginal_degree_data` builds one row per **participant-day** — zeros included, taken from the
participant roster rather than the contact table — in three weighting variants, at two resolutions:
the **marginal** degree of an adult / a child, and the degree split by the **contactee's** block as
well (`child→child`, `child→adult`, `adult→child`, `adult→adult`). It is run once per window in
`WINDOWS`.

⚠ **`cfg.w_dur_group` is `2.5/240`, exactly what `duration_weight` assigns a missing duration.** So
the "with group" variant is not a special convention, it is the ordinary one; the informative
contrast is against the variant that zeroes them.

⚠ **The four pairs do not sum to the marginals, and that is deliberate.** A pair needs the
contactee's age bin; contacts whose reported age falls entirely below the grid's first bin have none,
so they are dropped from the pairs and counted in the marginal's `unbinned` column. The marginals
were *not* recomputed on binned contacts — they are the observed degree distribution, and quietly
discarding part of it to make an identity tidier would change a published figure for a bookkeeping
convenience. The next cell asserts the identity in full, shortfall included.

⚠ For a pair, a zero is a participant-day with **no contact into that contactee block** — so every
pair keeps its whole block roster and the off-diagonal pairs are mostly zeros. That is real, and it
is what the hurdle `p⁰` fits.

In [ ]:
# ONE raw read (the config cell's `raw`) now feeds §1–§3 as well as §4: `marginal_degree_data` takes
# `load_raw_contact_inputs()`'s `(; df_part, craw)` — the raw arrow contact table, the only one
# carrying the `cnt_age_est_*` columns the contactor→contactee pairs need. It covers exactly the same
# contact rows as the joined frame this used to read (`(part_wave_uid, date)` is unique in
# part_uk.arrow), so the `:adult`/`:child` marginals below are unchanged by the switch.
MDS = Dict(w => marginal_degree_data(cfg; grid = grid, date_from = w[1], date_to = w[2], raw = raw)
           for w in WINDOWS)

GROUPS     = (:adult, :child)     # marginals — every contact the participant reported
PAIRS      = _MD_PAIR_KEYS        # contactor → contactee: child→child, child→adult, adult→child, adult→adult
ALL_GROUPS = (GROUPS..., PAIRS...)

wg = [(w, g) for w in WINDOWS for g in ALL_GROUPS]
DataFrame(
    window           = [string(w[1], " … ", w[2])         for (w, g) in wg],
    group            = [String(MDS[w][g].group)           for (w, g) in wg],
    what             = [MDS[w][g].title                   for (w, g) in wg],
    participant_days = [MDS[w][g].n                       for (w, g) in wg],
    contacts         = [MDS[w][g].n_contacts              for (w, g) in wg],
    unbinned         = [MDS[w][g].n_contacts_unbinned     for (w, g) in wg],
    pct_group        = [round(100 * MDS[w][g].n_group_contacts / max(MDS[w][g].n_contacts, 1); digits = 1)
                        for (w, g) in wg],
    mean_count       = [round(mean(MDS[w][g].k);  digits = 3) for (w, g) in wg],
    max_count        = [maximum(MDS[w][g].k)                  for (w, g) in wg],
    mean_w_group     = [round(mean(MDS[w][g].zg); digits = 4) for (w, g) in wg],
    mean_w_nogroup   = [round(mean(MDS[w][g].zn); digits = 4) for (w, g) in wg],
    zero_count       = [round(MDS[w][g].n_zero_k / MDS[w][g].n; digits = 4) for (w, g) in wg],
    zero_w_nogroup   = [round(MDS[w][g].n_zero_n / MDS[w][g].n; digits = 4) for (w, g) in wg],
)

In [ ]:
# The weighting is monotone by construction: zeroing group contacts can only remove weight and can
# only create zeros. If either assertion trips, the group flag or the join has gone wrong.
for w in WINDOWS, g in ALL_GROUPS
    m = MDS[w][g]
    @assert m.n_zero_n >= m.n_zero_g "zeroing group weights reduced the zero count for $g in $w"
    @assert mean(m.zn) <= mean(m.zg) "zeroing group weights increased the mean for $g in $w"
end

# The four pairs account for every contact the marginals have EXCEPT those whose contactee age cannot
# be assigned to a CIS bin (reported entirely below the grid's first bin). That shortfall is counted,
# not hidden, so the identity below is exact — and it is the check that the contactee binning is
# joined to the right participant block. The roster check is the second half: a pair must keep ALL of
# its block's participant-days, since a day with no contact into the other block is a zero, not a
# missing row.
for w in WINDOWS, (b, mg) in ((1, :child), (2, :adult))
    m  = MDS[w][mg]
    ps = [MDS[w][Symbol(_MD_BLOCK_SYM[b], "_", c)] for c in _MD_BLOCK_SYM]
    @assert m.n_contacts == sum(p.n_contacts for p in ps) + m.n_contacts_unbinned "pair contacts + unbinned ≠ marginal for $mg in $w"
    @assert all(p.n == m.n for p in ps) "a pair lost participant-days relative to the $mg marginal in $w"
    @printf("%s %-6s marginal %7d contacts = pairs %7d + unbinnable %4d (%.2f%%);  roster %6d\n",
            w[1], mg, m.n_contacts, sum(p.n_contacts for p in ps), m.n_contacts_unbinned,
            100 * m.n_contacts_unbinned / m.n_contacts, m.n)
end
println("weighting monotonicity + pair/marginal contact identity OK for all windows")

**What the table shows, and it is the analysis plan's own claim made visible.** Group contacts are
about half the contact table, yet zeroing their duration weight moves the mean weighted degree by
under 2% and the zero fraction by well under a percentage point. That is because each group contact
carries weight `2.5/240 ≈ 0.0104` — "a group contact's contributions are significantly diminished by
contact duration weights". Where they *do* matter is the **tail**, and the participant-days whose only
contacts were group ones. So read the two weighted curves apart in the ccdf tail, not at the mode.

Note also `max_count`: CoMix's mass-contact reports run into the thousands. That is what the empirical
CV² below reacts to, and what the NegBin cannot follow.

## §2 — Fits

Both likelihoods match the forecasting framework rather than the 6j notebook, so the parameters mean
here what they mean in the fitted grid:

- **`model_NegBinDegree`** — plain `NegBin(m, k)` on the integer degree, zeros modelled directly. Not
  zero-inflated: `model_degree` is not, and the analysis plan writes `P = NegBin(μ, k)`.
- **`model_HurdleWeibullDegree`** — `p0` plus a Weibull over the strictly positive weighted degrees,
  parameterised by the **mean of the positive part** with the scale derived as `λ = μ / Γ(1 + 1/κ)`,
  mirroring `_cell_moments!`. So the incl-zero mean `(1 − p0)·μ` is exactly the `K1` that
  `_weibull_moments` returns — the quantity §4 plots.

Both are fitted to all six groups, so the four pairs get the same treatment as the marginals — and
the pair fits are the marginal-degree counterpart of the block-linear dispersion the grid estimates
per week.

Chains go to `../res/14j_fit_*_<group>_<from>_<to>.jld2`; the summary table covers both windows.

In [ ]:
FITS = Dict{Any,Dict{Symbol,NamedTuple}}()
SUM  = DataFrame[]
for w in WINDOWS
    FITS[w] = Dict{Symbol,NamedTuple}()
    for g in ALL_GROUPS
        FITS[w][g] = fit_marginal_models(MDS[w][g], cfg; res_dir = "../res", n_sample = 2000)
        s = copy(FITS[w][g].summary)
        s[!, :window] .= string(w[1], " … ", w[2])          # so the two windows are comparable in one table
        push!(SUM, s)
    end
end
SUMMARY = select(vcat(SUM...), :window, :)
CSV.write("../res/14j_fit_summary.csv", SUMMARY)
SUMMARY

In [ ]:
# One chain per fit, so no R̂ across chains — `is_chains_converged` applies its ESS/R̂ rule to the
# single chain, which is the same check 6j uses for this strand.
for w in WINDOWS, g in ALL_GROUPS, (nm, ch) in (("negbin", FITS[w][g].negbin),
                                                ("hweibull", FITS[w][g].hweibull))
    @printf("%s %-12s %-9s converged=%-5s  min ESS=%.0f\n", w[1], g, nm, is_chains_converged(ch),
            minimum(skipmissing(MCMCChains.ess(ch).nt.ess)))
end

# The fits must reproduce the moments they were fitted to. NegBin's `m` is the mean of the counts;
# the hurdle-Weibull's `(1-p0)*mu` is the INCL-ZERO mean of the weighted-with-group degree.
#
# ⚠ The 5% tolerance is ASSERTED on the two marginals only, which is where it was measured. The four
# pairs are printed and warned above 10% instead: the off-diagonal pairs are ~90% zeros with a heavy
# positive tail, and a threshold nobody has measured on that shape has no business failing the
# notebook. A warning here is a signal to look at the panel, not a defect.
for w in WINDOWS, g in ALL_GROUPS
    s = FITS[w][g].summary
    getq(fam, q) = only(@subset(s, :family .== fam, :quantity .== q).median)
    m_hat  = getq("negbin", "m (mean)")
    k1_hat = getq("hweibull", "incl-zero mean (1-p0)*mu")
    md_    = MDS[w][g]
    e_nb, e_hw = m_hat/mean(md_.k) - 1, k1_hat/mean(md_.zg) - 1
    @printf("%s %-12s NegBin m %.4f vs %.4f (%+.2f%%) | (1-p0)mu %.5f vs %.5f (%+.2f%%)\n",
            w[1], g, m_hat, mean(md_.k), 100*e_nb, k1_hat, mean(md_.zg), 100*e_hw)
    if g in GROUPS
        @assert abs(e_nb) < 0.05
        @assert abs(e_hw) < 0.05
    elseif max(abs(e_nb), abs(e_hw)) > 0.10
        @warn "$(g) in $(w[1]) reproduces its moments to worse than 10% — inspect the §3 panel" e_nb e_hw
    end
end

## §3 — The degree-distribution figures

Two panels per group per window, both log–log, each carrying the three empirical series and the
two fitted curves with 90% posterior bands — for the two marginals and the four contactor →
contactee blocks.

**Everything is normalised over all participant-days, zeros included**, and the survival function is
`P(degree > x)`. That is deliberate: the fitted overlays are unconditional too — `ccdf(NegBin, k)` and
`(1 − p0)·ccdf(Weibull, z)` — so the two sit on one axis with no zero-truncation gymnastics, and the
height of the leftmost point of each ccdf *is* its non-zero fraction.

⚠ A note on the pdf panel's units. The count series is a pmf on unit-width integer bins, which is a
density, so the three series are dimensionally comparable — but they are **not on comparable levels**,
because the weighted degrees live on a support a few hundred times narrower and their density is
correspondingly higher. Compare each series against its own fitted curve, and compare the three with
each other by *shape*; read levels off the ccdf panel.

The y-range of each panel is set by the **empirical** series and the fitted curves are clipped into
it. Without that, the fitted NegBin evaluated out to a degree of several thousand returns densities
around 1e-40 and compresses all four real series into the top few percent of the panel.

⚠ Figure filenames carry the window — without it the second window would silently overwrite the first.

The **pairs figure** stacks the four blocks one per row, in the order child→child, child→adult,
adult→child, adult→adult. Read it for assortativity: the two diagonal rows should carry most of the
contacts, and child→child should be the row that moves when schools reopen.

In [ ]:
# Per-group panel pairs: the two marginals, then the four contactor → contactee blocks.
for w in WINDOWS, g in ALL_GROUPS
    display(make_degree_fig(MDS[w][g], FITS[w][g]; res_dir = "../res"))
end

In [ ]:
# The publication figures, one of each per window: the 2×2 marginal figure (adults top, children
# bottom) and the 4×2 pairs figure (one row per contactor → contactee block).
for w in WINDOWS
    display(make_degree_combined_fig(MDS[w], FITS[w]; res_dir = "../res"))
    display(make_degree_pairs_fig(MDS[w], FITS[w]; res_dir = "../res"))
end

**What to look for.** In the ccdf panels the observed count (solid black) has a far heavier tail
than the fitted NegBin (dashed black) can follow — which is the premise of this whole project, and the
reason the empirical CV² in `14j_fit_summary.csv` is an order of magnitude above the fitted one. The
two duration-weighted series lie almost on top of each other until the far tail, where zeroing group
contacts pulls the purple curve in. Comparing the two windows shows how much of the July–December
picture is carried by the final quarter alone.

**In the pairs figure**, the two diagonal rows (child→child, adult→adult) should dominate — contact
is strongly assortative in age — and the two off-diagonal rows sit far lower and start much further
down the y-axis, because their leftmost ccdf height *is* their non-zero fraction. `child→adult` is the
one row where the counts and the duration weighting disagree most: children report many adult
contacts, but the school-type contacts that dominate `child→child` are short, so the weighted curves
separate from the count curve differently in the two rows. That contrast is what the block-linear
dispersion in `model_degree` is fitted per block to absorb.

## §4 — Fitted contact mean over time, by age block

For each origin, the Stage-1 chain is read at **week column `cfg.n_fit`** — the origin week. Since
`-w8h` every horizon's degree window is anchored at `t₀ − n_fit + 1`, so t₀ sits at column `n_fit` in
*every* chain regardless of `h` (the same constant 10j asserts as `t_o_est == cfg.n_fit`).

The per-cell quantity is the incl-zero raw first moment `K1`, matching `_weibull_moments`:

$$\text{NegBin: } K_1[i,j] = \mu[i,j] \qquad\qquad \text{hurdle-Weibull: } K_1[i,j] = (1-p^0[i,j])\,\mu[i,j]$$

the total for contactor bin `i` is $\sum_j K_1[i,j]$ — "combining all contact means for each
contactor" — and the block value is the **population-weighted** mean of those totals over the bins in
the block. μ already carries `log(pop_j / pop_ref)`, so the row sum is contacts per participant.

⚠ Both levels are summarised **per draw**, so a block's 90% band is the band of the aggregate, not a
combination of the per-bin bands — the bins within a block are strongly correlated through the shared
GP, and combining their quantiles would overstate the spread. The seven-bin detail is not lost; it is
written to `14j_contact_mean_timeline_bins.csv`.

⚠ `contacts` is passed explicitly as `contacts_label(cfg)`, never left to the `CONTACTS_TOKEN`
default — that default is a compile-time constant built from the default `FrameworkConfig`, and
leaving it in place on readers like these is the bug that blanked five 10j figures and all eight of
9j's transmission panels.

### The observed overlay

`observed_contact_means` supplies the raw comparator, from the **model's own pipeline** — same age
binning, same duration weights, same roster-derived zeros — so the overlay is like-for-like rather
than an independently-computed number that would differ for uninteresting reasons. That costs 0.35%
of contacts (3061 of 865108) whose contactee age cannot be assigned to a CIS bin.

⚠ **The observed weighted series carries the `(1 − p⁰)` factor**, i.e. it is
`Σ_j (1 − p0[t,i,j]) · mean(positive weighted degrees)`, the incl-zero mean. It is deliberately NOT
10j's `_observed_cell_mean(…; weighted = true)`, which returns the positive-part mean alone — the
right comparator for **μ**, but not for the `K1 = (1−p⁰)·μ` plotted here. Without the factor the
overlay would sit about 4× high (p⁰ ≈ 0.75 per cell) and read as a badly under-fitting model rather
than a units mismatch.

### Phase shading

The bands are 9j's `PERIODS`/`shade_periods!` — the Munday-2023 Table 2 boundaries that
`plot_wis_diff_over_time` uses — **reused rather than restated**, so the two notebooks cannot
disagree about when a phase began. `PERIODS` runs 2020-11-05 … 2021-11-24, so the first ~3 and last
~5 origins of this grid fall outside every named phase and are legitimately unshaded, exactly as in
9j. `shade_periods!` must be called *after* the series (its docstring: a leading numeric overlay
collapses the Date axis), which is what `_timeline_panel!` does.

### The `p0f` guard

`_p0_draws_valid` rejects any chain in which a `p0f` coordinate is **identically 0 or 1 across every
draw**. That cannot be a Beta posterior under the hurdle Binomial likelihood — a cell the data push to
the boundary still returns varying values around 1e-4, never thousands of bit-identical zeros — so the
test is exact and cannot false-positive.

It exists because one chain (`weighted-hweibull @ 2021-02-14 h1`, the last file of an interrupted
transfer) once arrived with 61 of its 441 `p0f` coordinates unwritten. Since `K1 = (1−p0)·μ`, each dead
cell contributed its full `μ` and the child total at that origin inflated ×2.5 — which read as a real
signal and was not. Nothing else caught it: the file was normal size (`isfile` passed), the dimensions
were right (§0's structural check passed) and the parameter names were right (the generation gates
passed). **The current grid is clean, so the excluded list below should be empty**; the guard is a
regression net, since the grid arrives by transfer and the failure is silent.

### The third panel: excess (neighbourhood) degree

Panels 1–2 plot `MeanNGM`'s per-capita contact `C₀ = ⟨k⟩`. Panel 3 plots
`NeighbourhoodDegreeNGM`'s, from the **same** hurdle-Weibull Stage-1 draws:

$$C_0^{\text{nbhd}}[i,j] \;=\; \frac{\langle k^2\rangle}{\langle k\rangle}\cdot g
  \;=\; \frac{K_2[i,j]}{K_1[i,j]}\,(1-p^0[i,j]),
\qquad K_2 = (1-p^0)\,\mu^2\,(1+\mathrm{CV}^2_W),\quad
\mathrm{CV}^2_W = \frac{\Gamma(1+2/\kappa)}{\Gamma(1+1/\kappa)^2}-1$$

so the two builders that the forecasting grid is a 2×2 of are shown side by side over the whole
timeline. The excess degree is the **size-biased** mean — the expected degree of the person at the
far end of a randomly chosen contact — which is why it exceeds the mean whenever the degree
distribution has any spread at all, by exactly the factor `1 + CV²`.

⚠ **Nothing here is re-derived.** The reconstruction calls `_weibull_moments` (`joint_model.jl`) for
`(⟨k⟩, ⟨k²⟩, g)` and `base_contact(NeighbourhoodDegreeNGM(), …)` (`ngm.jl`) for the functional — the
framework's own functions, in the order `_cell_moments!` calls them. A hand-written
`(1−p⁰)·μ·(1+CV²)` here would be correct today and would silently stop tracking `base_contact` the
first time it changed. κ comes from `reconstruct_dispersion_draws`, which already mirrors
`_cell_moments!`'s `exp(softclamp(·, −4.3, 5))` and reads the same chain in the same draw order.

⚠ The **NegBin** path has an excess degree too (`_negbin_moments`, with `g = 1/(1−P₀)`), and it is
deliberately not plotted: the question this panel answers is what the neighbourhood NGM is fed on the
**duration-weighted** path, where the heavy tail lives.

⚠ Panel 3's y-axis is capped at the 99th percentile of the observed (never below the fitted band).
The observed excess is an empirical **second** moment over positive weighted degrees, so the same
single mass-contact report that already distorts the per-bin mean dominates it far harder after
squaring. Points pushed off-scale are counted in the panel title, never dropped silently.

In [ ]:
TL = Dict(d => collect_origin_contact_means(d, cfg; grid = grid, h = 1, origins = ORIGINS,
                                            contacts = contacts_label(cfg))
          for d in (NegBinAgePair(), HurdleWeibullAgePair()))
BINS     = vcat((t.bins     for t in values(TL))...)
BLOCKS   = vcat((t.blocks   for t in values(TL))...)
EXCLUDED = vcat((t.excluded for t in values(TL))...)

# Raw observed values, from the model's own pipeline. ONE prepare_degree_data call spans every
# origin week (see the docstring) rather than 63 separate windows.
OBS = observed_contact_means(cfg; grid = grid, origins = ORIGINS, raw = raw)

# `quantity` ("K1" / "excess") is half of every key now: the weighted model contributes two series
# per origin × bin, so joining on `degree` alone would silently duplicate every row.
CSV.write("../res/14j_contact_mean_timeline_bins.csv",              # 7-bin detail + its observed
          leftjoin(BINS, select(OBS.bins, [:degree, :quantity, :origin, :bin, :observed]),
                   on = [:degree, :quantity, :origin, :bin]))
if nrow(EXCLUDED) == 0
    println("p0f / dispersion guards: nothing excluded — every chain is a genuine posterior")
else
    @warn "guards excluded $(nrow(EXCLUDED)) origin-series — those chains need re-copying or refitting"
    display(EXCLUDED)
end
combine(groupby(BLOCKS, [:degree, :quantity]), nrow => :rows,
        :origin => (x -> length(unique(x))) => :origins,
        :origin => minimum => :first, :origin => maximum => :last)

In [ ]:
# INVARIANT: excess ≥ K1, everywhere, fitted and observed. The ratio is `1 + CV²` for the fitted
# series and `E[Z²]/E[Z]²` for the observed one, both ≥ 1 by construction, and both hold per draw and
# per cell — so they survive the row sum over contactee bins, the positive population weighting into
# blocks, and (being a monotone relation) the median across draws. Cheap, exact, and it catches a
# wrong `g`, a swapped K1/K2, or a Stage-1 chain whose κ draws are out of step with its μ draws.
for (nm, F, O, k) in (("blocks", BLOCKS, OBS.blocks, [:degree, :origin, :block_label]),
                      ("bins",   BINS,   OBS.bins,   [:degree, :origin, :bin]))
    pair(df, val) = innerjoin(select(@subset(df, :degree .== "weighted-hweibull", :quantity .== "excess"),
                                     k..., val => :ex),
                              select(@subset(df, :degree .== "weighted-hweibull", :quantity .== "K1"),
                                     k..., val => :k1), on = k)
    f, o = pair(F, :med), pair(O, :observed)
    @assert nrow(f) > 0 && nrow(o) > 0 "no weighted excess/K1 pairs to check in $nm"
    @assert all(f.ex .>= f.k1 .- 1e-9) "fitted excess < K1 in $nm — check g / the K1,K2 order"
    @assert all(o.ex .>= o.k1 .- 1e-9) "observed excess < K1 in $nm"
    @printf("%-6s n=%4d  excess/K1  fitted median %5.2f (max %6.2f) | observed median %5.2f (max %7.2f)\n",
            nm, nrow(f), median(f.ex ./ f.k1), maximum(f.ex ./ f.k1),
            median(o.ex ./ o.k1), maximum(o.ex ./ o.k1))
end

In [ ]:
# Headline: two lines (child / adult) per panel. The block CSV is written by this call, with
# `observed` joined in on (degree, quantity, origin, block_label).
make_contact_mean_timeline_fig(BLOCKS; obs = OBS.blocks, grid = grid, res_dir = "../res")

In [ ]:
# Per-age-group twin: the same three panels, one line per CIS bin, so it is visible which age
# groups drive the block lines. Thinner ribbons and smaller markers because there are 14 series per
# panel; the y-axis is capped at the 99th percentile of the observed (never below the fitted bands)
# and any points pushed off-scale are counted in the panel title rather than silently dropped.
make_contact_mean_bins_fig(BINS; obs = OBS.bins, grid = grid, res_dir = "../res")

**What to look for.** The observed `×` should scatter around the fitted line rather than sit
systematically to one side — the GP smooths week-to-week sampling noise, so the fit is the trend and
the markers are the raw weekly means, including weeks whose roster is small enough to be very noisy.

The two mean panels should step up at the **"Lockdown 3 Schools open"** band, when English schools
reopened: the children block roughly doubles on the unweighted path and rises by about a quarter on
the weighted one. That difference is the point of duration weighting — school contacts are numerous
but short, so weighting by duration damps exactly the contacts the count measure is dominated by. A
weighted panel that tracked the unweighted one step-for-step would be the suspicious result.

**The excess-degree panel is on a different scale, and should be**: it is the mean multiplied by
`1 + CV²`, and the whole premise of this project is that the weighted degree distribution has a heavy
tail, i.e. a CV² well above 1. Read it against panel 2 — the *ratio* between them is the size-biasing
factor the neighbourhood NGM applies, and where that ratio moves over time is where the two NGM
builders would disagree about transmission even given identical mean contacts. The previous cell
asserts the ratio never drops below 1.

⚠ **The excess panel's fitted and observed series do not track each other as closely as the two mean
panels do** — the invariant cell above prints a median `excess/K1` of about 2.3 fitted against about
1.8 observed. That gap is expected, and it is worth reading rather than fixing. `K1` is the quantity
the hurdle-Weibull effectively targets; the excess degree depends on the fitted **second** moment,
which no part of the likelihood is asked to reproduce. On top of that the shape `κ` is block-linear —
one value per child/adult block per week, shared across all 49 ordered cells — so a single parameter
carries the whole block's tail behaviour, while the observed excess is computed cell by cell.

So read this panel as **what the neighbourhood NGM is actually fed**, not as a goodness-of-fit check.
Whether that input forecasts better than the mean is the question the 2×2 grid exists to answer, and
9j answers it by WIS, not by eye.

## §5 — Outputs

Written to `../res`:

| file | what |
|---|---|
| `14j_grid_audit.csv` | per-cell audit of every Stage-1 artefact |
| `14j_degree_<group>_<from>_<to>.png` | per-group pdf + ccdf panels — six groups × two windows |
| `14j_degree_combined_<from>_<to>.png` | the 2×2 marginal publication figure, one per window |
| `14j_degree_pairs_<from>_<to>.png` | the 4×2 contactor → contactee figure, one per window |
| `14j_fit_negbin_<group>_<from>_<to>.jld2` | NegBin chain + `n_obs` / `n_zero` / `window` |
| `14j_fit_hweibull_<group>_<from>_<to>.jld2` | hurdle-Weibull chain + `n_pos` as well |
| `14j_fit_summary.csv` | fitted medians and 90% intervals with the empirical moments beside them, all six groups, both windows |
| `14j_contact_mean_timeline.png` / `.csv` | §4's block figure and the table behind it, fitted + observed, `quantity ∈ {K1, excess}` |
| `14j_contact_mean_timeline_bins.png` / `.csv` | the per-age-group twin and its table |

To add a further window, append to `WINDOWS` in the config cell — every filename downstream is keyed
on it, so nothing overwrites anything. To add a further §4 panel, append to `_TL_SPECS` in
`14j_viz_utils.jl`: both figures derive their layout from it.